## 3DoF Entry VTOL w/o Aoa SCP

Imports

In [ ]:
# Basic imports
import importlib
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Trajopt imports
import trajopt; importlib.reload(trajopt)
import trajopt.core.modules.method.scp as scp
import trajopt.core.problem as prob
import trajopt.utils.config_loader as cfg
import trajopt.core.modules.analysis.standalone as standalone

create problem and run SCP

In [ ]:
example_name = "vtol1_entry_3dof"
nominal_config  = cfg.load_configs(example_name)

# create problem instance
problem = prob.Problem(nominal_config)

# run SCP
problem = scp.run_scp(problem)

# store scenario data struct for plotting
scenario_data = standalone.run_standalone_analysis(problem)

run SCP

make plots

In [ ]:
# --- 3D Interactivity (run %matplotlib widget in Jupyter) ---
#%matplotlib widget # ← run this once per notebook session

mission = problem.mission
model   = problem.model
method  = problem.method

# --- Retrieve trajectory and parameters ---

t_init = method.t_init
z_init = method.z_init
nu_init = method.nu_init
x_init  = z_init[:, 0]
y_init  = z_init[:, 1]
z_init  = z_init[:, 2]

t_opt  = problem.solution["ts"]
z_opt  = problem.solution["zs"]
nu_opt  = problem.solution["us"]

nd = method.nondim['nd']
nt = method.nondim['nt']
nm = method.nondim['nm']

plt.plot(t_opt * nt, (z_opt[:, 0] * nd - mission.planet['r']) / 1e3)
plt.ylabel('altitude [km]')
plt.xlabel('time [s]')

plt.figure()
plt.plot(t_opt*method.nondim['nt'], (z_opt[:, 3] * method.nondim['nv']))
plt.ylabel('velocity [km/s]')
plt.xlabel('time [s]')

plt.figure()
plt.plot(t_opt*method.nondim['nt'], np.rad2deg(nu_opt[:, 0]))
plt.ylabel('bank angle [deg]')
plt.ylim(-80, 80)
plt.xlabel('time [s]')

# Extract original state (first n columns) when CTCS is enabled
n = problem.model.n
z_state = problem.solution["zs"][:, :n]

# Get state variables
rs = z_state[:, 0]  # radius
vs = z_state[:, 3]  # velocity

# Compute density and dynamic pressure
rho = np.array([problem.mission.mission_module.atmosphere_model_jax(r, problem) for r in rs])
nv = method.nondim["nv"]
q_dyn = 0.5 * rho * (vs * nv) ** 2

# Plot
plt.figure()
plt.plot(t_opt * method.nondim['nt'], q_dyn)
plt.ylabel('dynamic pressure [Pa]')
plt.xlabel('time [s]')
plt.grid(True)

print(f"final time: {t_opt[-1] * method.nondim['nt']}")

print(f"final state: {method.nondim['M']['state']['nd2d'] @ z_opt[-1, :model.n]}")


# testing nonlinear propagation matches with scp solution
soln = scenario_data["method"]["mc_data"][0]["iters"][-1]

print(soln.keys())

plt.figure()
plt.scatter(soln['t_nl'], (soln['z_nl'][:, 0] - mission.planet['r']) / 1e3)
plt.scatter(t_opt * nt, (z_opt[:, 0] * nd - mission.planet['r']) / 1e3)
plt.ylabel('altitude [km]')
plt.xlabel('time [s]')

plt.figure()
plt.plot(soln['t_nl'], soln['u_nl'])
plt.scatter(t_opt * nt, nu_opt[:, 0] * method.nondim['nang'])
plt.ylabel('bank angle [deg]')
plt.xlabel('time [s]')
plt.legend()


In [ ]:
# Extract original state (first n columns) when CTCS is enabled
n = problem.model.n
z_state = problem.solution["zs"][:, :n]

# Get longitude (theta) and latitude (phi) from state
# State indices: x_idx=1 (longitude), y_idx=2 (latitude)
theta = z_state[:, 1]  # longitude
phi = z_state[:, 2]    # latitude

# Convert to degrees if needed (check if they're in radians)
import numpy as np
theta_deg = np.rad2deg(theta) if np.max(np.abs(theta)) < 10 else theta
phi_deg = np.rad2deg(phi) if np.max(np.abs(phi)) < 10 else phi

# Get NFZ parameters
xc_deg = np.array(problem.mission.obs["xc"])
yc_deg = np.array(problem.mission.obs["yc"])
rc_deg = np.array(problem.mission.obs["rc"])

# Plot trajectory
plt.figure(figsize=(10, 8))
plt.plot(theta_deg, phi_deg, 'b-', linewidth=2, label='Trajectory', zorder=2)
plt.plot(theta_deg[0], phi_deg[0], 'go', markersize=10, label='Start', zorder=3)
plt.plot(theta_deg[-1], phi_deg[-1], 'ro', markersize=10, label='End', zorder=3)

# Plot no-fly zones as circles
for i in range(len(xc_deg)):
    circle = plt.Circle((xc_deg[i], yc_deg[i]), rc_deg[i], color='red', fill=True, alpha=0.4, 
                       edgecolor='darkred', linewidth=2, label='No-Fly Zone' if i == 0 else '', zorder=1)
    plt.gca().add_patch(circle)
    # Also plot center point
    plt.plot(xc_deg[i], yc_deg[i], 'rx', markersize=12, markeredgewidth=2, zorder=4)

# Set axis limits to include both trajectory and NFZ circles
x_min = min(np.min(theta_deg), np.min(xc_deg - rc_deg)) - 2
x_max = max(np.max(theta_deg), np.max(xc_deg + rc_deg)) + 2
y_min = min(np.min(phi_deg), np.min(yc_deg - rc_deg)) - 2
y_max = max(np.max(phi_deg), np.max(yc_deg + rc_deg)) + 2

plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)

plt.xlabel('Longitude [deg]', fontsize=12)
plt.ylabel('Latitude [deg]', fontsize=12)
plt.title('Trajectory with No-Fly Zones', fontsize=14)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()